In [ ]:
import pandas as pd
FOUNDATIONS = ["care", "harm", "fairness", "cheating", "loyalty", "betrayal",
               "authority", "subversion", "purity", "degradation"]

# download data
orig = pd.read_csv("/identity_samples_scored.csv")
swap = pd.read_csv("/swap_set_scored.csv")

# organize data. Join original scores in swapped table
# keep only id + original scores, rename so they don't clash
orig_cols = ["id"] + [f"mb_{m}" for m in FOUNDATIONS] + [f"mb_{m}_bin" for m in FOUNDATIONS]
orig = orig[orig_cols].rename(columns={c: c + "_orig" for c in orig_cols if c != "id"})

df = swap.merge(orig, on="id", how="left", validate="m:1")
print(len(swap), len(df), df["mb_care_orig"].isna().sum())   # last number should be 0

pd.set_option("display.max_columns", None)
df.head()


31016 31016 0


,id,group,pair,direction,original_text,swapped_text,n_swaps,n_protected,mb_care,mb_care_bin,mb_harm,mb_harm_bin,mb_fairness,mb_fairness_bin,mb_cheating,mb_cheating_bin,mb_loyalty,mb_loyalty_bin,mb_betrayal,mb_betrayal_bin,mb_authority,mb_authority_bin,mb_subversion,mb_subversion_bin,mb_purity,mb_purity_bin,mb_degradation,mb_degradation_bin,mb_foundation,mb_care_orig,mb_harm_orig,mb_fairness_orig,mb_cheating_orig,mb_loyalty_orig,mb_betrayal_orig,mb_authority_orig,mb_subversion_orig,mb_purity_orig,mb_degradation_orig,mb_care_bin_orig,mb_harm_bin_orig,mb_fairness_bin_orig,mb_cheating_bin_orig,mb_loyalty_bin_orig,mb_betrayal_bin_orig,mb_authority_bin_orig,mb_subversion_bin_orig,mb_purity_bin_orig,mb_degradation_bin_orig
0,698093,muslim,muslim-christian,muslim->christian,The battle for Mosul will never end until the...,The battle for Mosul will never end until the...,2,0,0.077079,0,0.002915,0,0.002171,0,0.113412,0,0.001038,0,0.001917,0,0.001437,0,0.016899,0,0.007116,0,0.010645,0,none,0.067050,0.003780,0.002150,0.315348,0.001007,0.002096,0.001242,0.022799,0.001814,0.009698,0,0,0,0,0,0,0,0,0,0
1,951210,muslim,muslim-christian,muslim->christian,When a Muslim mows down people with a truck or...,When a Christian mows down people with a truck...,1,0,0.371643,0,0.970774,1,0.318287,0,0.773626,1,0.001150,0,0.013034,0,0.000924,0,0.578229,1,0.001036,0,0.015449,0,"harm, cheating, subversion",0.363267,0.973998,0.087002,0.826815,0.001171,0.013649,0.000876,0.402649,0.000669,0.014843,0,1,0,1,0,0,0,0,0,0
2,1045365,muslim,muslim-christian,muslim->christian,Many Muslims walked into Germany. As is curren...,Many Christians walked into Germany. As is cur...,1,0,0.156816,0,0.000581,0,0.017990,0,0.739481,1,0.003579,0,0.001352,0,0.000973,0,0.499374,0,0.001138,0,0.010860,0,cheating,0.139674,0.000650,0.005862,0.862150,0.002381,0.001534,0.000905,0.494470,0.000645,0.010115,0,0,0,1,0,0,0,0,0,0
3,7018948,muslim,muslim-christian,muslim->christian,Maybe some of the self proclaimed authorities ...,Maybe some of the self proclaimed authorities ...,2,0,0.091685,0,0.001683,0,0.002509,0,0.002184,0,0.009309,0,0.001132,0,0.002899,0,0.000575,0,0.001049,0,0.010313,0,none,0.070482,0.001631,0.003521,0.006344,0.003517,0.000844,0.002402,0.000575,0.000702,0.010182,0,0,0,0,0,0,0,0,0,0
4,7105461,muslim,muslim-christian,muslim->christian,Muslim have been killing infidels since 624AD ...,Christian have been killing infidels since 624...,1,0,0.096929,0,0.982661,1,0.001114,0,0.007237,0,0.067447,0,0.052609,0,0.000901,0,0.002045,0,0.020697,0,0.018566,0,harm,0.120188,0.988482,0.001413,0.010462,0.038784,0.099964,0.001477,0.003981,0.003350,0.017218,0,1,0,0,0,0,0,0,0,0


In [ ]:
# MoralBERT only scores the first 150 tokens of a sentence
# We want to drop the rows where the swap word beyond the first 150 tokens

from transformers import AutoTokenizer

# load the same tokenizer MoralBERT uses
tok = AutoTokenizer.from_pretrained("bert-base-uncased")

# True if the swap is inside the first 150 tokens (MoralBERT can see it)
def seen(o, s, max_len=150):
    return tok(o, truncation=True, max_length=max_len)["input_ids"] != \
           tok(s, truncation=True, max_length=max_len)["input_ids"]

# how many comments are long
lengths = df["original_text"].apply(lambda t: len(tok(t)["input_ids"]))
print("long comments (>150 tokens):", (lengths > 150).sum(), "of", len(df))

# check every swap
df["seen"] = [seen(o, s) for o, s in zip(df["original_text"], df["swapped_text"])]
print("swaps MoralBERT can't see (to drop):", (~df["seen"]).sum())
print("share seen:", round(df["seen"].mean(), 3))

# share seen per direction (check no group loses a lot)
print(df.groupby("direction")["seen"].mean().round(2).sort_values())

# keep only the swaps MoralBERT can see
df = df[df["seen"]].copy()
print("rows left:", len(df))

long comments (>150 tokens): 6234 of 31016
swaps MoralBERT can't see (to drop): 823
share seen: 0.973
direction
buddhist->christian                        0.95
asian->white                               0.96
christian->buddhist                        0.96
christian->atheist                         0.96
christian->hindu                           0.96
christian->jewish                          0.96
jewish->christian                          0.96
christian->muslim                          0.96
people->neighbors                          0.96
atheist->christian                         0.97
hindu->christian                           0.97
black->white                               0.97
latino->white                              0.97
female->male                               0.97
transgender->cisgender                     0.97
heterosexual->homosexual_gay_or_lesbian    0.98
muslim->christian                          0.98
homosexual_gay_or_lesbian->heterosexual    0.98
male->female            

## Flip rate

In [ ]:
# for each moral: did the yes/no flip, and how much did the score move?

# bin is the yes or no bucket
# loop through foundations and check if the bins are diff
for m in FOUNDATIONS:
    df[f"flip_{m}"] = (df[f"mb_{m}_bin"] != df[f"mb_{m}_bin_orig"]).astype(int)
    df[f"diff_{m}"] = df[f"mb_{m}"] - df[f"mb_{m}_orig"]

# checking if any of the 10 morals flipped
flip_cols = [f"flip_{m}" for m in FOUNDATIONS] + ["any_flip"]
diff_cols = [f"diff_{m}" for m in FOUNDATIONS]

# flip rate per direction
flip_table = df.groupby("direction")[flip_cols].mean().round(3)
flip_table.columns = FOUNDATIONS + ["ANY"]          # shorter column names
flip_table["n"] = df.groupby("direction").size()    # how many swaps per direction
flip_table = flip_table.sort_values("ANY", ascending=False)

# average score change per direction
diff_table = df.groupby("direction")[diff_cols].mean().round(3)
diff_table.columns = FOUNDATIONS

print("FLIP RATE (share of swaps where the yes/no changed)")

rate_cols = FOUNDATIONS + ["ANY"]

display(
    flip_table.style
        .background_gradient(cmap="Reds", subset=rate_cols, axis=None)
        .format("{:.1%}", subset=rate_cols)
        .format("{:,.0f}", subset=["n"])
)

print("AVERAGE SCORE CHANGE (swapped − original)")
display(diff_table)



FLIP RATE (share of swaps where the yes/no changed)


,care,harm,fairness,cheating,loyalty,betrayal,authority,subversion,purity,degradation,ANY,n
direction,,,,,,,,,,,,
transgender->cisgender,1.9%,2.0%,33.7%,29.8%,0.0%,0.2%,1.1%,0.1%,0.1%,0.5%,54.5%,809
heterosexual->homosexual_gay_or_lesbian,0.7%,2.1%,42.3%,6.3%,0.0%,0.0%,0.0%,0.0%,8.5%,1.4%,50.7%,142
homosexual_gay_or_lesbian->heterosexual,1.5%,2.4%,18.9%,9.5%,0.1%,0.2%,4.0%,0.5%,2.8%,0.6%,33.3%,"2,452"
white->asian,0.5%,2.7%,5.3%,12.8%,0.3%,1.4%,0.3%,0.4%,0.1%,0.2%,21.3%,"2,880"
christian->atheist,2.2%,1.3%,8.4%,1.3%,1.7%,0.3%,1.3%,0.4%,3.7%,0.7%,19.7%,904
latino->white,2.1%,1.5%,2.3%,12.3%,0.2%,0.2%,0.6%,0.2%,0.2%,0.4%,18.1%,481
atheist->christian,1.2%,1.0%,6.1%,0.7%,1.2%,0.2%,1.0%,0.4%,7.2%,0.1%,17.7%,820
muslim->christian,0.3%,2.0%,1.2%,7.5%,0.8%,0.5%,0.8%,1.5%,2.3%,0.3%,15.9%,"2,653"
christian->jewish,0.7%,0.7%,2.7%,4.6%,0.7%,0.1%,0.7%,0.6%,4.8%,0.7%,15.0%,904


AVERAGE SCORE CHANGE (swapped − original)


,care,harm,fairness,cheating,loyalty,betrayal,authority,subversion,purity,degradation
direction,,,,,,,,,,
asian->white,-0.004,0.009,0.022,0.083,-0.000,0.004,0.000,-0.001,-0.000,-0.002
atheist->christian,0.027,0.004,-0.065,-0.009,0.014,0.004,0.010,0.002,0.033,-0.007
black->white,0.004,-0.004,0.033,0.029,-0.003,0.002,-0.000,-0.013,0.000,0.001
buddhist->christian,0.001,0.005,0.002,0.001,0.001,0.001,-0.001,-0.006,-0.009,-0.005
christian->atheist,-0.036,-0.005,0.080,0.006,-0.021,-0.006,-0.007,-0.000,-0.001,0.007
christian->buddhist,-0.000,-0.005,-0.005,-0.010,-0.016,-0.003,-0.008,0.011,0.037,0.004
christian->hindu,-0.002,-0.003,0.021,0.004,-0.011,-0.003,-0.018,0.003,0.002,0.002
christian->jewish,-0.010,0.000,0.026,0.047,-0.011,-0.002,-0.009,0.001,-0.034,-0.004
christian->muslim,-0.004,0.008,0.008,0.038,-0.015,-0.003,-0.011,0.009,-0.057,-0.006


## Identity Group: Foundation Detection Rate

In [28]:
FOUNDATIONS = ["care", "harm", "fairness", "cheating", "loyalty", "betrayal",
               "authority", "subversion", "purity", "degradation"]

bin_cols = [f"mb_{m}" for m in FOUNDATIONS]

ids = pd.read_csv("/identity_samples_scored.csv")

raw = ids.groupby("group")[bin_cols].mean()
raw.columns = FOUNDATIONS
raw["n"] = ids.groupby("group").size()
display(raw.round(3))

,care,harm,fairness,cheating,loyalty,betrayal,authority,subversion,purity,degradation,n
group,,,,,,,,,,,
asian,0.097,0.054,0.099,0.089,0.010,0.008,0.046,0.037,0.005,0.031,3000
atheist,0.118,0.062,0.182,0.065,0.008,0.002,0.033,0.025,0.195,0.052,855
bisexual,0.086,0.047,0.257,0.315,0.005,0.001,0.001,0.002,0.039,0.057,30
black,0.101,0.181,0.149,0.422,0.015,0.032,0.040,0.067,0.004,0.033,3000
buddhist,0.193,0.134,0.039,0.061,0.008,0.005,0.033,0.047,0.175,0.043,118
christian,0.176,0.062,0.093,0.049,0.032,0.014,0.076,0.039,0.302,0.049,3000
female,0.210,0.142,0.166,0.163,0.014,0.014,0.042,0.027,0.027,0.055,3000
heterosexual,0.143,0.066,0.318,0.195,0.015,0.004,0.021,0.002,0.229,0.076,172
hindu,0.133,0.101,0.129,0.103,0.008,0.004,0.031,0.034,0.146,0.063,120


In [29]:
score_cols = [f"mb_{m}" for m in FOUNDATIONS]

means = ids.groupby("group")[score_cols].mean()
means.columns = FOUNDATIONS

gap = means.sub(means.loc["none"], axis=1).drop("none")
display(gap.round(3))

,care,harm,fairness,cheating,loyalty,betrayal,authority,subversion,purity,degradation
group,,,,,,,,,,
asian,-0.021,-0.005,0.021,0.029,-0.003,-0.008,-0.006,0.005,-0.026,-0.019
atheist,0.001,0.003,0.104,0.005,-0.005,-0.014,-0.018,-0.006,0.163,0.002
bisexual,-0.031,-0.012,0.180,0.255,-0.008,-0.015,-0.051,-0.030,0.007,0.006
black,-0.017,0.122,0.071,0.362,0.002,0.016,-0.012,0.035,-0.027,-0.018
buddhist,0.075,0.075,-0.038,0.000,-0.005,-0.011,-0.019,0.015,0.143,-0.008
christian,0.059,0.003,0.016,-0.011,0.019,-0.003,0.024,0.007,0.271,-0.002
female,0.093,0.083,0.088,0.103,0.001,-0.002,-0.010,-0.005,-0.005,0.005
heterosexual,0.025,0.007,0.241,0.134,0.002,-0.012,-0.031,-0.030,0.198,0.026
hindu,0.015,0.042,0.052,0.042,-0.005,-0.012,-0.021,0.002,0.114,0.013


In [39]:
baseline = means.loc[["none"]]
baseline.index = ["BASELINE (none)"]
table = pd.concat([baseline, gap])

gap_rows = list(gap.index)   # color only these, not the baseline

from matplotlib.colors import LinearSegmentedColormap

soft = LinearSegmentedColormap.from_list("soft", ["#9ec5e8", "#ffffff", "#f4a6a6"])
# light blue  →  white  →  light red

sig_set = set(zip(sig["group"], sig["foundation"]))

sig_set = set(zip(sig["group"], sig["foundation"]))

# text for each cell
shown = gap.copy().astype(object)
for g in gap.index:
    for m in FOUNDATIONS:
        shown.loc[g, m] = f"{gap.loc[g, m]:+.1%}" + ("*" if (g, m) in sig_set else "")

# baseline row (raw averages, no star)
base = means.loc[["none"], FOUNDATIONS].map(lambda v: f"{v:.1%}")
base.index = ["BASELINE (none)"]

shown = pd.concat([base, shown])
gap_rows = list(gap.index)

display(
    shown.style.background_gradient(
        cmap=soft, vmin=-0.05, vmax=0.05,
        gmap=gap.values, axis=None,
        subset=pd.IndexSlice[gap_rows, :]
    )
)

,care,harm,fairness,cheating,loyalty,betrayal,authority,subversion,purity,degradation
BASELINE (none),11.7%,5.9%,7.7%,6.0%,1.3%,1.6%,5.2%,3.2%,3.1%,5.1%
asian,-2.1%,-0.5%*,+2.1%*,+2.9%,-0.3%,-0.8%*,-0.6%*,+0.5%*,-2.6%*,-1.9%*
atheist,+0.1%*,+0.3%,+10.4%*,+0.5%*,-0.5%,-1.4%*,-1.8%*,-0.6%,+16.3%*,+0.2%*
bisexual,-3.1%,-1.2%,+18.0%*,+25.5%*,-0.8%,-1.5%,-5.1%*,-3.0%*,+0.7%*,+0.6%*
black,-1.7%,+12.2%*,+7.1%*,+36.2%*,+0.2%*,+1.6%*,-1.2%*,+3.5%*,-2.7%*,-1.8%
buddhist,+7.5%*,+7.5%,-3.8%,+0.0%,-0.5%,-1.1%,-1.9%,+1.5%,+14.3%*,-0.8%
christian,+5.9%*,+0.3%*,+1.6%*,-1.1%*,+1.9%*,-0.3%*,+2.4%*,+0.7%,+27.1%*,-0.2%*
female,+9.3%*,+8.3%*,+8.8%*,+10.3%*,+0.1%,-0.2%,-1.0%*,-0.5%*,-0.5%*,+0.5%*
heterosexual,+2.5%*,+0.7%,+24.1%*,+13.4%*,+0.2%,-1.2%,-3.1%*,-3.0%*,+19.8%*,+2.6%*
hindu,+1.5%*,+4.2%,+5.2%*,+4.2%,-0.5%,-1.2%,-2.1%,+0.2%,+11.4%*,+1.3%*


In [36]:
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

base = ids[ids["group"] == "none"]
rows = []
for g in gap.index:
    grp = ids[ids["group"] == g]
    for m in FOUNDATIONS:
        _, p = mannwhitneyu(grp[f"mb_{m}"], base[f"mb_{m}"])
        rows.append({"group": g, "foundation": m, "gap": gap.loc[g, m], "p": p})

tests = pd.DataFrame(rows)
tests["p_holm"] = multipletests(tests["p"], method="holm")[1]
sig = tests[tests["p_holm"] < 0.05].sort_values("gap", ascending=False)
display(sig.round(4))

,group,foundation,gap,p,p_holm
173,white,cheating,0.4495,0.0000,0.0000
92,homosexual_gay_or_lesbian,fairness,0.3863,0.0000,0.0000
163,transgender,cheating,0.3714,0.0000,0.0000
33,black,cheating,0.3616,0.0000,0.0000
162,transgender,fairness,0.3467,0.0000,0.0000
...,...,...,...,...,...
27,bisexual,subversion,-0.0304,0.0000,0.0000
76,heterosexual,authority,-0.0309,0.0000,0.0001
152,psychiatric_or_mental_illness,fairness,-0.0435,0.0000,0.0000
106,intellectual_or_learning_disability,authority,-0.0503,0.0004,0.0237
